# NB01 - SGD & Momentum
## Background
Gradient descent is the engine behind all modern deep learning. Before 2012, researchers debated whether stochastic gradient descent (SGD) could train deep networks at all. Momentum methods, borrowed from classical optimization theory (Polyak 1964, Nesterov 1983), dramatically stabilized training by accumulating a velocity vector that smooths noisy gradient estimates.

## What You'll Learn
- Vanilla SGD, SGD+Momentum, and Nesterov Accelerated Gradient (NAG) from scratch
- The geometry of momentum: why it helps in ravines
- Optimizer comparison on convex (quadratic) and non-convex (Rosenbrock) problems

## Why This Matters
Even with Adam dominating modern training, SGD+Momentum often finds flatter minima that generalize better (Keskar et al. 2017). Understanding its mechanics lets you diagnose oscillation and know when to reach for it over adaptive methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Callable, List, Tuple, Optional

@dataclass
class SGD:
    lr: float = 0.01
    momentum: float = 0.0
    nesterov: bool = False
    _v: Optional[np.ndarray] = field(default=None, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._v is None:
            self._v = np.zeros_like(params)
        if self.nesterov:
            self._v = self.momentum * self._v + grads
            return params - self.lr * (grads + self.momentum * self._v)
        else:
            self._v = self.momentum * self._v + grads
            return params - self.lr * self._v

    def reset(self):
        self._v = None

def quadratic(x: np.ndarray) -> Tuple[float, np.ndarray]:
    loss = x[0]**2 + 10 * x[1]**2
    grad = np.array([2*x[0], 20*x[1]])
    return loss, grad

def rosenbrock(x: np.ndarray) -> Tuple[float, np.ndarray]:
    a, b = 1.0, 100.0
    loss = (a - x[0])**2 + b * (x[1] - x[0]**2)**2
    dx = -2*(a - x[0]) - 4*b*x[0]*(x[1] - x[0]**2)
    dy = 2*b*(x[1] - x[0]**2)
    return loss, np.array([dx, dy])

def run_optimizer(opt, fn: Callable, x0: np.ndarray, n_steps: int = 500):
    x = x0.copy().astype(float)
    opt.reset()
    losses, trajectory = [], [x.copy()]
    for _ in range(n_steps):
        loss, grad = fn(x)
        losses.append(loss)
        x = opt.step(x, grad)
        trajectory.append(x.copy())
    return losses, trajectory

x0 = np.array([-1.5, 0.8])
configs = [
    ('SGD lr=0.05',         SGD(lr=0.05)),
    ('SGD+Momentum m=0.9',  SGD(lr=0.05, momentum=0.9)),
    ('Nesterov m=0.9',      SGD(lr=0.05, momentum=0.9, nesterov=True)),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, opt in configs:
    losses, traj = run_optimizer(opt, quadratic, x0, n_steps=300)
    axes[0].semilogy(losses, label=label)
    traj = np.array(traj)
    axes[1].plot(traj[:, 0], traj[:, 1], '-o', markersize=2, label=label)

axes[0].set_title('Convergence on Ill-Conditioned Quadratic')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Loss (log scale)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Optimization Trajectory')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
axes[1].scatter([0], [0], c='red', s=100, zorder=5, label='Optimum')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_01_quadratic.png', dpi=100, bbox_inches='tight')
plt.show()
print('Quadratic comparison done.')

In [ ]:
x0_rb = np.array([-1.0, 1.0])
configs_rb = [
    ('SGD lr=0.001',        SGD(lr=0.001)),
    ('SGD+Momentum m=0.9',  SGD(lr=0.001, momentum=0.9)),
    ('Nesterov m=0.9',      SGD(lr=0.001, momentum=0.9, nesterov=True)),
]

fig, ax = plt.subplots(figsize=(10, 5))
results = {}
for label, opt in configs_rb:
    losses, _ = run_optimizer(opt, rosenbrock, x0_rb, n_steps=5000)
    ax.semilogy(losses, label=label)
    results[label] = losses[-1]

ax.set_title('Convergence on Rosenbrock (Non-Convex)')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss (log scale)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_01_rosenbrock.png', dpi=100, bbox_inches='tight')
plt.show()

print('Final losses after 5000 steps:')
for label, final_loss in results.items():
    print(f'  {label:35s}: {final_loss:.6f}')

print()
print('Momentum mechanics summary:')
print('  v_t = beta*v_{t-1} + g_t')
print('  theta_t = theta_{t-1} - lr*v_t')
print('  Effective step (beta=0.9): 1/(1-0.9) = 10x amplification in consistent direction')